In [1]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, f1_score

In [2]:
df = pd.read_csv("../data/bangla_news_articles.csv")

In [3]:
SEED = 42
tf.keras.utils.set_random_seed(SEED)

In [4]:
df = df[["text", "category"]].dropna()
df["text"] = df["text"].astype(str).str.strip()

In [5]:
df = df.drop_duplicates(subset="text")

In [6]:
MIN_COUNT = 20
counts = df["category"].value_counts()
keep = counts[counts >= MIN_COUNT].index
df = df[df["category"].isin(keep)].reset_index(drop=True)

In [7]:
le = LabelEncoder()
y = le.fit_transform(df["category"])
num_classes = len(le.classes_)
print(dict(zip(le.classes_, np.bincount(y))))

{'crime': np.int64(161), 'economy': np.int64(57), 'education': np.int64(34), 'entertainment': np.int64(25), 'health': np.int64(22), 'international': np.int64(53), 'national': np.int64(411), 'other': np.int64(905), 'politics': np.int64(334), 'sports': np.int64(42)}


In [8]:
X = df["text"].values
X_trainval, X_test, y_trainval, y_test = train_test_split(
    X, y, test_size=0.15, stratify=y, random_state=SEED)
X_train, X_val, y_train, y_val = train_test_split(
    X_trainval, y_trainval, test_size=0.15, stratify=y_trainval, random_state=SEED)

In [9]:
def bangla_standardize(text):
    return tf.strings.regex_replace(text, r"[।,.!?;:\"'()\[\]\-]", " ")

MAX_TOKENS = 30000
SEQ_LEN = 300

vectorizer = layers.TextVectorization(
    max_tokens=MAX_TOKENS,
    output_sequence_length=SEQ_LEN,
    standardize=bangla_standardize,
)
vectorizer.adapt(X_train)

In [12]:
def to_input(x):
    x = [str(t) for t in x]
    return tf.reshape(tf.constant(x, dtype=tf.string), (-1, 1))

In [13]:
EMB_DIM = 128
LSTM_UNITS = 64



model = models.Sequential([
    layers.Input(shape=(1,), dtype=tf.string),
    vectorizer,
    layers.Embedding(input_dim=MAX_TOKENS, output_dim=EMB_DIM, mask_zero=True),
    layers.Bidirectional(layers.LSTM(LSTM_UNITS, dropout=0.2)),
    layers.Dense(64, activation="relu"),
    layers.Dropout(0.3),
    layers.Dense(num_classes, activation="softmax"),
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)
model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ text_vectorization                   │ (None, 300)                 │               0 │
│ (TextVectorization)                  │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ embedding_1 (Embedding)              │ (None, 300, 128)            │       3,840,000 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ bidirectional_1 (Bidirectional)      │ (None, 128)                 │          98,816 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_2 (Dense)                      │ (None, 64)                  │           8,256 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_1 (Dropout)                  │ (None, 64)                  │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_3 (Dense)                      │ (None, 10)                  │             650 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 3,947,722 (15.06 MB)

 Trainable params: 3,947,722 (15.06 MB)

 Non-trainable params: 0 (0.00 B)

In [14]:
cw = compute_class_weight("balanced", classes=np.unique(y_train), y=y_train)
class_weight = dict(enumerate(cw))

early = callbacks.EarlyStopping(
    monitor="val_loss", patience=3, restore_best_weights=True)

history = model.fit(
    to_input(X_train), y_train,
    validation_data=(to_input(X_val), y_val),
    epochs=20,
    batch_size=32,
    class_weight=class_weight,
    callbacks=[early],
)

Epoch 1/20
47/47 ━━━━━━━━━━━━━━━━━━━━ 47s 655ms/step - accuracy: 0.1030 - loss: 2.3013 - val_accuracy: 0.1188 - val_loss: 2.2920
Epoch 2/20
47/47 ━━━━━━━━━━━━━━━━━━━━ 27s 565ms/step - accuracy: 0.1484 - loss: 2.2022 - val_accuracy: 0.1992 - val_loss: 2.2523
Epoch 3/20
47/47 ━━━━━━━━━━━━━━━━━━━━ 40s 545ms/step - accuracy: 0.2534 - loss: 1.7499 - val_accuracy: 0.2107 - val_loss: 1.9616
Epoch 4/20
47/47 ━━━━━━━━━━━━━━━━━━━━ 27s 568ms/step - accuracy: 0.4892 - loss: 0.9999 - val_accuracy: 0.2950 - val_loss: 1.7229
Epoch 5/20
47/47 ━━━━━━━━━━━━━━━━━━━━ 41s 573ms/step - accuracy: 0.7703 - loss: 0.4258 - val_accuracy: 0.3257 - val_loss: 1.8791
Epoch 6/20
47/47 ━━━━━━━━━━━━━━━━━━━━ 41s 576ms/step - accuracy: 0.8902 - loss: 0.2356 - val_accuracy: 0.3602 - val_loss: 2.0184
Epoch 7/20
47/47 ━━━━━━━━━━━━━━━━━━━━ 41s 571ms/step - accuracy: 0.9512 - loss: 0.1039 - val_accuracy: 0.3870 - val_loss: 2.3483


In [15]:
probs = model.predict(to_input(X_test), verbose=0)
y_pred = probs.argmax(axis=1)

majority = np.bincount(y_test).argmax()
print("Çoğunluk sınıfı baseline accuracy:", (y_test == majority).mean())
print("Model accuracy:", (y_test == y_pred).mean())
print("Macro F1:", f1_score(y_test, y_pred, average="macro"))
print(classification_report(y_test, y_pred,
                            target_names=le.classes_, zero_division=0))

Çoğunluk sınıfı baseline accuracy: 0.44299674267100975
Model accuracy: 0.3355048859934853
Macro F1: 0.2448036236818541
               precision    recall  f1-score   support

        crime       0.29      0.42      0.34        24
      economy       0.21      0.44      0.29         9
    education       0.33      0.40      0.36         5
entertainment       0.00      0.00      0.00         4
       health       0.00      0.00      0.00         3
international       0.17      0.25      0.20         8
     national       0.42      0.31      0.36        62
        other       0.62      0.24      0.34       136
     politics       0.26      0.64      0.37        50
       sports       0.13      0.33      0.19         6

     accuracy                           0.34       307
    macro avg       0.24      0.30      0.24       307
 weighted avg       0.44      0.34      0.33       307



In [16]:
def predict_category(texts):
    p = model.predict(to_input(texts), verbose=0)
    return le.inverse_transform(p.argmax(axis=1))

print(predict_category([df["text"].iloc[0]]))

['crime']
